In [0]:
%python
import csv
import io
import re
from decimal import Decimal
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DecimalType
)

# 1. Configuration
sql_file_path = "/Volumes/banking/source/raw_data/03_Incrementat_data.sql"#"/Volumes/banking/source/raw_data/historical_data_insert_script.sql"
catalog_name = "banking"
schema_name = "source"

# 2. Define parsing schemas matching the columns in your INSERT INTO statements
schemas = {
    "branches": StructType([
        StructField("branch_code", StringType(), False),
        StructField("branch_name", StringType(), False),
        StructField("city", StringType(), False),
        StructField("state", StringType(), False),
        StructField("region", StringType(), True),
    ]),
    "customers": StructType([
        StructField("customer_id", IntegerType(), False),
        StructField("first_name", StringType(), False),
        StructField("last_name", StringType(), False),
        StructField("date_of_birth", StringType(), False),
        StructField("pan_number", StringType(), True),
        StructField("email", StringType(), True),
        StructField("phone_number", StringType(), True),
        StructField("kyc_status", StringType(), True),
        StructField("branch_code", StringType(), False),
    ]),
    "accounts": StructType([
        StructField("account_id", LongType(), False),
        StructField("customer_id", IntegerType(), False),
        StructField("account_type", StringType(), False),
        StructField("balance", DecimalType(18, 2), False),
        StructField("currency", StringType(), True),
        StructField("branch_code", StringType(), False),
        StructField("status", StringType(), True),
        StructField("opened_date", StringType(), False),
    ]),
    "transactions": StructType([
        StructField("txn_id", LongType(), False),
        StructField("account_id", LongType(), False),
        StructField("txn_type", StringType(), False),
        StructField("amount", DecimalType(18, 2), False),
        StructField("txn_timestamp", StringType(), False),
        StructField("channel", StringType(), True),
        StructField("status", StringType(), True),
    ])
}

# 3. Read and parse the SQL file using csv.reader (no ast needed)
with open(sql_file_path, "r", encoding="utf-8") as f:
    sql_content = f.read()

# Strip SQL block comments /* ... */ and line comments -- ...
sql_content = re.sub(r"/\*.*?\*/", "", sql_content, flags=re.DOTALL)
sql_content = re.sub(r"--.*?$", "", sql_content, flags=re.MULTILINE)

insert_pattern = re.compile(
    r"INSERT\s+INTO\s+(?:\w+\.|#)?(?:stg_)?(\w+)\s*\([^)]+\)\s*VALUES\s*(.*?);",
    re.IGNORECASE | re.DOTALL
)
tuple_pattern = re.compile(r"\(([^()]+)\)")

table_rows = {tbl: [] for tbl in schemas}

for match in insert_pattern.finditer(sql_content):
    tbl = match.group(1).lower()
    if tbl not in table_rows:
        continue
        
    values_block = match.group(2)
    for t_match in tuple_pattern.finditer(values_block):
        row_str = t_match.group(1).strip()
        # Parse comma-separated SQL values respecting single quotes
        reader = csv.reader(io.StringIO(row_str), quotechar="'", skipinitialspace=True)
        raw_vals = next(reader)
        
        # Convert 'NULL' strings to None
        vals = [None if v.upper() == "NULL" else v for v in raw_vals]
        
        # Cast numeric fields to match StructType
        if tbl == "customers":
            vals[0] = int(vals[0])
        elif tbl == "accounts":
            vals[0] = int(vals[0])
            vals[1] = int(vals[1])
            vals[3] = Decimal(vals[3])
        elif tbl == "transactions":
            vals[0] = int(vals[0])
            vals[1] = int(vals[1])
            vals[3] = Decimal(vals[3])
            
        table_rows[tbl].append(tuple(vals))

# 4. Cast dates/timestamps, append audit columns, and insert in strict FK order
for tbl in ["branches", "customers", "accounts", "transactions"]:
    rows = table_rows[tbl]
    if not rows:
        print(f"No rows found for {tbl}")
        continue
        
    df = spark.createDataFrame(rows, schema=schemas[tbl])
    
    if tbl == "branches":
        df = df.withColumn("created_at", F.current_timestamp())
    elif tbl == "customers":
        df = (df
              .withColumn("date_of_birth", F.to_date("date_of_birth", "yyyy-MM-dd"))
              .withColumn("created_at", F.current_timestamp())
              .withColumn("updated_at", F.current_timestamp()))
    elif tbl == "accounts":
        df = (df
              .withColumn("opened_date", F.to_date("opened_date", "yyyy-MM-dd"))
              .withColumn("created_at", F.current_timestamp())
              .withColumn("updated_at", F.current_timestamp()))
    elif tbl == "transactions":
        df = (df
              .withColumn("txn_timestamp", F.to_timestamp("txn_timestamp", "yyyy-MM-dd HH:mm:ss"))
              .withColumn("created_at", F.current_timestamp()))
        
    target_table = f"{catalog_name}.{schema_name}.{tbl}"
    df.write.mode("append").insertInto(target_table)
    print(f"Successfully inserted {len(rows)} rows into {target_table}")

Successfully inserted 5 rows into banking.source.branches
Successfully inserted 1500 rows into banking.source.customers
Successfully inserted 1500 rows into banking.source.accounts
Successfully inserted 5000 rows into banking.source.transactions
